# Select Subregion size for images across views

Based on: A Transfer Learning Method for Meteorological Visibility Estimation Based on Feature Fusion Method

The original work used a single camera angle. Our goal is to use multiple camera angles. As such, we will need a way to handle varying sizes of "useable" pixels in each image. The work focuses on selecting regions of interest based off of a grey normal distribution. Whenthere is only one camera angle, all of the subregions will be of the same size. With multiple camera angles, each angle will have a different size region. This notebook will explore approaches as to how to handle this when preparing for model training.

In [1]:
import os
import cv2
import sqlite3
import numpy as np
import json
from matplotlib import pyplot as plt
from pathlib import Path
from fogvis.db import ImageEntity, Database
from scipy.stats import norm 
from dataclasses import dataclass, asdict
from concurrent.futures import ProcessPoolExecutor
from fogvis.preprocessing import calculate_subregion_for_images, SubregionRequest, SubregionResult

#assuming default import media location
GLOBAL_CANDIDATE_T_VALUES = [230, 235]
DB_PATH : Path = Path(os.path.abspath(os.path.join(os.getcwd(), os.pardir, os.pardir, "media", "db", "database.sqlite3")))
PRE_MASK_GREY_VALUE_CLIP = 254
NUM_SUBREGIONS_TO_SELECT = 5
MAX_WORKERS = 12

In [2]:
def get_all_camera_ids_for_scene(db : Database, scene_name : str) -> list[int]: 
    ids = []

    with db.get_connection() as cur: 
        scene_sql : str = f"SELECT id FROM scene WHERE NAME = '{scene_name}'"
        scene_id:int = cur.execute(scene_sql).fetchone()[0]

        cam_sql : str = f"SELECT id FROM camera WHERE sceneID = {scene_id}"
        result = cur.execute(cam_sql).fetchall()
        for r in result: 
            ids.append(r[0])

    return ids

db = Database(DB_PATH)
c_ids = get_all_camera_ids_for_scene(db, "colorado")
c_full_ids = get_all_camera_ids_for_scene(db, "colorado_full")
c_ids.extend(c_full_ids)

def get_images_for_one_camera_angle(db : Database, camera_ids : list[int]) -> dict[int, list[Path]]:
    image_paths : dict[int, list[Path]] = {}

    with db as conn: 
        sql = f"SELECT filePath from image WHERE cameraID = ?"

        for id in camera_ids: 
            params = (id,)
            result = conn.execute(sql, params).fetchall()

            if result is None: 
                raise Exception()
            
            image_paths[id] = []
            for r in result: 
                name = r[0]
                fPath : Path = Path(os.path.join(Path(db.db_path).parent, "images", name))
                image_paths[id].append(fPath)

    return image_paths


camera_images : dict[int, list[Path]] = get_images_for_one_camera_angle(db, c_ids)
camera_images

{1: [WindowsPath('e:/Repos/FoggyVision/media/db/images/2026-04-11_10-08-52_Frame-0.png'),
  WindowsPath('e:/Repos/FoggyVision/media/db/images/2026-04-11_10-08-52_Frame-1.png'),
  WindowsPath('e:/Repos/FoggyVision/media/db/images/2026-04-11_10-08-52_Frame-10.png'),
  WindowsPath('e:/Repos/FoggyVision/media/db/images/2026-04-11_10-08-52_Frame-11.png'),
  WindowsPath('e:/Repos/FoggyVision/media/db/images/2026-04-11_10-08-52_Frame-12.png'),
  WindowsPath('e:/Repos/FoggyVision/media/db/images/2026-04-11_10-08-52_Frame-13.png'),
  WindowsPath('e:/Repos/FoggyVision/media/db/images/2026-04-11_10-08-52_Frame-14.png'),
  WindowsPath('e:/Repos/FoggyVision/media/db/images/2026-04-11_10-08-52_Frame-15.png'),
  WindowsPath('e:/Repos/FoggyVision/media/db/images/2026-04-11_10-08-52_Frame-16.png'),
  WindowsPath('e:/Repos/FoggyVision/media/db/images/2026-04-11_10-08-52_Frame-17.png'),
  WindowsPath('e:/Repos/FoggyVision/media/db/images/2026-04-11_10-08-52_Frame-18.png'),
  WindowsPath('e:/Repos/FoggyVi

In [ ]:
requests : list[SubregionRequest] = []
for id in camera_images: 
    requests.append(SubregionRequest(id, camera_images[id], NUM_SUBREGIONS_TO_SELECT, GLOBAL_CANDIDATE_T_VALUES))

camera_subregions = {}
failed_ids = []
with ProcessPoolExecutor(max_workers=MAX_WORKERS) as executor: 
    for result in executor.map(calculate_subregion_for_images, requests): 
        if result.subregions is None: 
            failed_ids.append(result.camera_id)
        else:
            camera_subregions[result.camera_id] = result.subregions

failed_ids

[1441,
 1442,
 1443,
 1444,
 1445,
 1446,
 1447,
 1448,
 1449,
 1450,
 1451,
 1452,
 1453,
 1454,
 1455,
 1456,
 1457,
 1458,
 1459,
 1460,
 1461,
 1462,
 1463,
 1464,
 1465,
 1466,
 1467,
 1468,
 1469,
 1470,
 1471,
 1472,
 1473,
 1474,
 1475,
 1476,
 1477,
 1478,
 1479,
 1480,
 1481,
 1482,
 1483,
 1484,
 1485,
 1486,
 1487,
 1488,
 1489,
 1490,
 1491,
 1492,
 1493,
 1494,
 1495,
 1496,
 1497,
 1498,
 1499,
 1500,
 1501,
 1502,
 1503,
 1504,
 1505,
 1506,
 1507,
 1508,
 1509,
 1510,
 1511,
 1512,
 1513,
 1514,
 1515,
 1516,
 1517,
 1518,
 1519,
 1520,
 1521,
 1522,
 1523,
 1524,
 1525,
 1526,
 1527,
 1528,
 1529,
 1530,
 1531,
 1532,
 1533,
 1534,
 1535,
 1536,
 1537,
 1538,
 1539,
 1540,
 1541,
 1542,
 1543,
 1544,
 1545,
 1546,
 1547,
 1548,
 1549,
 1550,
 1551,
 1552,
 1553,
 1554,
 1555,
 1556,
 1557,
 1558,
 1559,
 1560,
 1561,
 1562,
 1563,
 1564,
 1565,
 1566,
 1567,
 1568,
 1569,
 1570,
 1571,
 1572,
 1573,
 1574,
 1575,
 1576,
 1577,
 1578,
 1579,
 1580,
 1581,
 1582,
 1583,

In [4]:
failed_ids

1800